In [1]:
import os
import sys

project_root = "/root/work/tvm-ansor"
os.environ["TVM_HOME"] = f"{project_root}"
os.environ["TVM_LIBRARY_PATH"] = f"{project_root}/build-release"
if f"{project_root}/python" not in sys.path:
    sys.path.insert(0, f"{project_root}/python")

sys.path = [p for p in sys.path if not p.startswith(f"{project_root}/build")]
sys.path.append(f"{project_root}/build-release")
os.environ["LD_LIBRARY_PATH"] = f"{project_root}/build-release:" + os.environ.get("LD_LIBRARY_PATH", "")


import numpy as np
from util_manager import PathManager, get_network
import tvm
from tvm import auto_scheduler

TARGET = tvm.target.Target("cuda")

def get_tasks(mod, params, path_manager, verbose=True, get_pkl=True):
    if get_pkl:
        tasks, task_weights = path_manager.tasks_pkl_use()
    
    if get_pkl is False or tasks is None:
        print("Extract tasks...")
        tasks, task_weights = auto_scheduler.extract_tasks(mod["main"], params, TARGET)
        if path_manager.tasks_pkl_check() is False:
            path_manager.tasks_pkl_save(tasks, task_weights)

    if verbose:
        for idx, task in enumerate(tasks):
            print("========== Task %d : %s  (workload key: %s) ==========" % (idx, task.desc, task.workload_key))
            print(task.compute_dag)
    
    print(f"Total tasks length : {len(tasks)}")
    return tasks, task_weights

[15:40:41] /root/work/tvm-ansor/src/target/target_kind.cc:181: Warning: Unable to detect CUDA version, default to "-arch=sm_50" instead


In [17]:
from types import SimpleNamespace

args = SimpleNamespace(
    network="resnet_18",
    batch_size=1,
    dtype="float32",
    layout="NHWC",
    timenow=None,
    json=None
)

mod, params, input_shape, output_shape = get_network(args.network, args.batch_size, args.layout, dtype=args.dtype)
path_manager = PathManager(args.network, input_shape, args, None, None)
tasks, task_weights = get_tasks(None, params, path_manager, verbose=False, get_pkl=True)
tasks, task_weights = zip(*sorted(zip(tasks, task_weights), key=lambda x: x[0].desc))

task_idx_by_wk = {task.workload_key : idx for idx, task in enumerate(tasks)}

search_policies = []
for idx, (task, weight) in enumerate(zip(tasks, task_weights)):
    print(f"T{idx} : {task.desc} ({weight})")
    search_policies.append(
        auto_scheduler.SketchPolicy(task, auto_scheduler.XGBModel(), verbose=False)
    )

Getting network resnet_18...

Using json : /root/work/tvm-ansor/gallery/logs_json/resnet_18/(1,224,224,3)-0317_1551.json
Load tasks from /root/work/tvm-ansor/gallery/ansor_tasks_pkl/resnet_18-(1,224,224,3).pkl
Total tasks length : 24
T0 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add (1)
T1 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_1 (1)
T2 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_2 (1)
T3 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_3 (1)
T4 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_add_nn_relu (1)
T5 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_add_nn_relu_1 (1)
T6 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_add_nn_relu_2 (1)
T7 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_multiply_add_nn_re_a4e88f85cf7823fc_ (1)
T8 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight

In [20]:
from tvm.auto_scheduler.measure import recover_measure_input
from tvm import auto_scheduler
inputs, results = auto_scheduler.RecordReader("/root/work/tvm-ansor/gallery/logs_json/resnet_18/resnet_18-B1.json").read_lines()
states = []
states_by_task_idx = {}
for inp in inputs:
    inp = recover_measure_input(inp, True)
    states.append(inp.state)
    task_idx = task_idx_by_wk[inp.task.workload_key]
    if task_idx not in states_by_task_idx:
        states_by_task_idx[task_idx] = []
    states_by_task_idx[task_idx].append(inp.state)

In [25]:
states_by_task_idx[8]

[Placeholder: p0, p1, p2
 data_pack auto_unroll: 16
 blockIdx.x p.0@ci.0@p.1@ci.1@.0 (0,392)
   threadIdx.x p.0@ci.0@p.1@ci.1@.1 (0,32)
     for eps (0,6)
       for nu (0,6)
         for p (blockIdx_x // 28 * 14 + (blockIdx_x * 4 + threadIdx_x) % 14,1)
           for ci ((blockIdx_x % 28 * 32 + threadIdx_x) // 14,1)
             input_tile = ...
     unroll eps (0,6)
       unroll nu (0,6)
         unroll r_a (0,6)
           unroll r_b (0,6)
             data_pack = ...
 blockIdx.x eps.0@nu.0@p.0@co.0@ (0,56)
   vthread eps.1@nu.1@p.1@co.1@ (0,12)
     threadIdx.x eps.2@nu.2@p.2@co.2@ (0,24)
       bgemm.local auto_unroll: 1024
       for eps_c.0 (0,1)
         for nu_c.0 (0,1)
           for p_c.0 (0,1)
             for co_c.0 (0,1)
               for eps_c.1 (0,1)
                 for nu_c.1 (0,1)
                   for p_c.1 (0,1)
                     for co_c.1 (0,1)
                       for eps_c.2 (0,1)
                         for nu_c.2 (0,1)
                           for 

In [11]:
inp.task.workload_key

'["1097323f3970e5c881ad3a0028ca79cb", [1, 14, 14, 256], [4, 4, 256, 256], [1, 1, 1, 256], [1, 14, 14, 256]]'

In [ ]:
states[0].

1